<a href="https://colab.research.google.com/github/alicsrsustain-sudo/HVAC-Optimization-/blob/main/pump_affinity_law2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#INPUTS
# ─────────────────────────────────────────────
CURRENT_SPEED_PCT   = 95          # % of full speed (current)
NEW_SPEED_PCT       = 85          # % of full speed (proposed)
PUMP_RATED_POWER_KW = 5.5        # Pump motor rated power in kW (nameplate)
MOTOR_EFFICIENCY    = 0.919        # Motor efficiency (0–1), typical 0.88–0.95
OPERATING_HOURS_PER_YEAR = 3120  # Hours the pump runs per year (8760 = 24/7)
ELECTRICITY_COST_PER_KWH = 0.2225   # £/kWh — UK average (adjust for your tariff)


# ─────────────────────────────────────────────
# CALCULATIONS
# ─────────────────────────────────────────────
def affinity_power_ratio(speed_current_pct: float, speed_new_pct: float) -> float:
    """Return the power ratio using the cube law: (N2/N1)^3"""
    return (speed_new_pct / speed_current_pct) ** 3


def shaft_power(rated_power_kw: float, speed_pct: float, efficiency: float) -> float:
    """Actual power drawn at a given speed (kW), accounting for motor efficiency."""
    # At partial speed, shaft power scales with cube law relative to rated
    # Rated power is typically at 100% speed; scale to actual speed first
    shaft_kw = rated_power_kw * (speed_pct / 100) ** 3
    # Electrical input = shaft power / motor efficiency
    electrical_kw = shaft_kw / efficiency
    return electrical_kw


# Current and new electrical power draw
power_current_kw = shaft_power(PUMP_RATED_POWER_KW, CURRENT_SPEED_PCT, MOTOR_EFFICIENCY)
power_new_kw     = shaft_power(PUMP_RATED_POWER_KW, NEW_SPEED_PCT,     MOTOR_EFFICIENCY)

# Power saved
power_saved_kw   = power_current_kw - power_new_kw
power_reduction_pct = (power_saved_kw / power_current_kw) * 100

# Annual energy (kWh)
energy_current_kwh = power_current_kw * OPERATING_HOURS_PER_YEAR
energy_new_kwh     = power_new_kw     * OPERATING_HOURS_PER_YEAR
energy_saved_kwh   = energy_current_kwh - energy_new_kwh

# Annual cost savings
cost_current  = energy_current_kwh * ELECTRICITY_COST_PER_KWH
cost_new      = energy_new_kwh     * ELECTRICITY_COST_PER_KWH
cost_saved    = cost_current - cost_new

# CO₂ savings (UK grid average ~0.233 kg CO₂/kWh — DESNZ 2024)
CO2_FACTOR_KG_PER_KWH = 0.233
co2_saved_kg  = energy_saved_kwh * CO2_FACTOR_KG_PER_KWH
co2_saved_tonnes = co2_saved_kg / 1_000


# ─────────────────────────────────────────────
# REPORT
# ─────────────────────────────────────────────
SEPARATOR = "=" * 55

print(SEPARATOR)
print("   HVAC PUMP SPEED REDUCTION — SAVINGS REPORT")
print(SEPARATOR)

print("\n  SYSTEM CONFIGURATION")
print(f"   Pump rated power          : {PUMP_RATED_POWER_KW:.1f} kW")
print(f"   Motor efficiency           : {MOTOR_EFFICIENCY*100:.0f}%")
print(f"   Annual operating hours     : {OPERATING_HOURS_PER_YEAR:,} h")
print(f"   Electricity tariff         : £{ELECTRICITY_COST_PER_KWH:.4f}/kWh")

print("\n  POWER CONSUMPTION")
print(f"   Current speed              : {CURRENT_SPEED_PCT}%  →  {power_current_kw:.2f} kW")
print(f"   Proposed speed             : {NEW_SPEED_PCT}%  →  {power_new_kw:.2f} kW")
print(f"   Power saved                : {power_saved_kw:.2f} kW  ({power_reduction_pct:.1f}% reduction)")

print("\n  ANNUAL ENERGY SAVINGS")
print(f"   Energy at current speed    : {energy_current_kwh:,.0f} kWh/year")
print(f"   Energy at proposed speed   : {energy_new_kwh:,.0f} kWh/year")
print(f"    Energy saved            : {energy_saved_kwh:,.0f} kWh/year")

print("\n  ANNUAL COST SAVINGS")
print(f"   Cost at current speed      : £{cost_current:,.2f}/year")
print(f"   Cost at proposed speed     : £{cost_new:,.2f}/year")
print(f"    Money saved             : £{cost_saved:,.2f}/year")

   HVAC PUMP SPEED REDUCTION — SAVINGS REPORT

  SYSTEM CONFIGURATION
   Pump rated power          : 5.5 kW
   Motor efficiency           : 92%
   Annual operating hours     : 3,120 h
   Electricity tariff         : £0.2225/kWh

  POWER CONSUMPTION
   Current speed              : 95%  →  5.13 kW
   Proposed speed             : 85%  →  3.68 kW
   Power saved                : 1.46 kW  (28.4% reduction)

  ANNUAL ENERGY SAVINGS
   Energy at current speed    : 16,009 kWh/year
   Energy at proposed speed   : 11,467 kWh/year
    Energy saved            : 4,542 kWh/year

  ANNUAL COST SAVINGS
   Cost at current speed      : £3,562.07/year
   Cost at proposed speed     : £2,551.46/year
    Money saved             : £1,010.61/year
